# Welcome to the CADRE JEDI-edu tutorial
***
In this tutorial series we will explore the basic features of several data assimilation  (DA) solvers, using a simple two-layer quasi-geostrophic (QG) toy model. The model mimics the evolution of large-scale features, such as high and low pressure cells in the atmosphere. In the following we first provide a short description of the quasi-geostrophic model and what it represents, as well as introduce the building blocks of DA - namely, the  background model and observations that will be used in all subsequent solver tutorials. 

## The Quasi-geostrophic model

This section provides a short description of the quasi-geostrophic model used in the experiments throughout this tutorial series. The model evolves **potential vorticity** over time, with a **velocity field** derived from the **stream function**. In turn, the stream function field is calculated from the potential vorticity. Mathematically, the model has two layers and for each layer the model equations are <br>
<center>$\frac{D q_i}{dt} = 0$ </center> <br>
where the $q_i$ denote the quasi-geostrophic potential vorticity in vertical layer $i$. The equation expresses that potential vorticity is conserved following air parcels. It is given by <br>
<center>$q_i = \nabla^2 \psi_i + \mathcal{F}_{i,i-1} (\psi_{i-1} -\psi_i) + \mathcal{F}_{i,i+1}(\psi_{i+1}-\psi_i) + \beta y$ </center> <br>
This equation describes the three terms that build up potential vorticity, namely relative vorticity, which is the rotation of air parcels relative to the solid Earth, the layer thickness, represented by the $\mathcal{F}$ terms, and the northward position of the air parcel y where $\beta$ is the northward derivative of the Coriolis parameter. This parameter describes the daily rotation of the Earth around it axis. The actual expressions for the $\mathcal{F}$ will not be given here, but, as you would expect, they contain the layer thicknesses and the potential temperatures of the layers. <br> <br>
The conservation equation for potential vorticity describes the balance of changes in relative vorticity, in planetary vorticity, and in stretching of the fluid column. When one or more of these term changes the others have to compensate to conserve potential vorticity. <br> <br>
The model represents a zonal strip around the Earth, so the fields are periodic in the East-West direction and we use free slip on the northern and southern boundaries (which means that the meridional velocities are zero along those boundaries, but the zonal velocities are free to vary). <br>

## Setting up for data-assimilation experiments

To set up the data assimilation experiments in subsequent tutorials we take the following steps:

1. We first generate the equivalent of a real atmospheric, oceanic, or other system. This run we call the **truth**, and in the following we will use the atmosphere as an example.
2. We then generate the **background state**, the starting point for our data assimilation experiments. This starting point will be close to, but not the same as, the truth starting point. To be consistent, we perturb the truth state with a random draw from the initial state error distribution (chosen as Gaussian). In the real world this background state comes from a forecast, and contains all prior information we have.
3. Then we generate **observations** from the truth run, just as observations are taken in the real world. In these simplified model simulation experiments, we take measurements of the truth run and **add an observation error** to that measurement. This observation error is a random draw from the error distribution of the observations. In most DA solvers such as 3Dvar, the assumption is that observation errors are uncorrelated and have Gaussian-distributed errors. 
4. The final step, performed in each of the subsequent tutorials, is to run the DA experiment directrly using the products produced from (1-3).
<br>
The following is a quick overview of where the files are. To do this, you can navigate the menu on your left or you can list the files under the different directories.<br><br>
<u>On using Jupyter Labs notebooks</u>: you can click on any box that contains code (like the one below) and hit play, it will run the piece of code there!
<br>
<center> <img src="images/run_command.png"/> </center>
<br>
<br>

For convenience, you _need_ to export a few environment variables:

In [ ]:
export JEDI_BIN=/home/nonroot/build/bin
export JEDI_EDU=/home/nonroot/shared/EDU

Now, try to run the following code:

In [ ]:
ls -RCd --color $JEDI_BIN/qg*            # list the files in this directory
ls -RCd --color $JEDI_EDU/qg3Dvar/*/*
#which python3
#python3 -c "import site; print(site.getsitepackages())"


<br>
Look at the directory structure you just displayed: this is where you will need to navigate to find plots, yamls, analysis et background files... <br><br>

The files you will change are the **yaml files** (see where they are in the structure above). When you run jedi it will create output as indicated. <u>Note that each experiment has a *bg* directory with the background files, an *obs* directory with the observation files and a *da* directory with the analysis files.</u> These files are generated by jedi and can  also be vizualized using python, as explained later.


Now you are all ready to go!

# Generate the Truth simulation 
***

The JEDI system works with **yaml files**, which are essentially "namelist" files that contain commands/settings that execute specific tasks coded in the JEDI system. We first generate a truth run, which would correspond to the evolution of the true atmosphere. This run will be the basis for all experiments we will do in this tutorial series.

We will use from the yaml-files directory the file `generate_truth.yaml`. Note the comments after each line, explaining the contents of each line in the yaml file.

First move to your directory `/home/nonroot/shared/EDU/qgtruth/yamls` and open (double click on the file in the tree) the file `generate_truth.yaml`, which looks like this:

```yaml
forecast length: P21D                             # Run truth model for 18 days
geometry:                                         # Declare the geometry of the model grid points
  nx: 40                                          # number of grid points in zonal direction
  ny: 20                                          # number of grid points in the meridional direction
  depths: [4000.0, 6000.0]                        # Height of 1st and 2nd layer
initial condition:                                # Declare details of initial condition
  date: 2009-12-15T00:00:00Z                      # Date of initial condition
  read_from_file: 0                               # Initial condition not read from file but internally generated
model:                                            # Declare model used
  name: QG                                        # Quasi-geostrophic model
  tstep: PT10M                                    # Time step is 10 min
output:                                           # Declare output
  datadir: qgstart/output/truth/                  # Directory where output file will be stored
  date: 2009-12-15T00:00:00Z                      # Date of output file
  exp: truth                                      # Experiment to generate truth file
  frequency: PT6H                                 # Output frequency is every 6 hours
  type: fc                                        # Truth comes from running qg model in forecast mode
prints:                                           # Declare output specifics
  frequency: PT3H                                 # Generate output every 3 hours
```

Check what this file does: the file runs the 40 by 20 by 2 (layers) quasi-geostrophic model in a standard configuration. Have a good look at the file, see if it makes sense, and note the simple instructions for this simple problem.


After this yaml file is edited as desired we can generate the truth run via running the below commands:

**Pro-tip**: The output logs from these jedi executable runs can get long and cumbersome to scroll past. You can collapse the output logs by right clicking and selecting "Enable scrolling for outputs".

In [ ]:
cd $JEDI_EDU/
$JEDI_BIN/qg_forecast.x ./qgstart/yamls/generate_truth.yaml

You should be able to find the output files in `/home/nonroot/shared/EDU/qgstart/output/truth`. The directory should have the files
```
truth.fc.2009-12-15T00:00:00Z.PT0S.nc
truth.fc.2009-12-15T00:00:00Z.PT6H.nc
truth.fc.2009-12-15T00:00:00Z.PT12H.nc
truth.fc.2009-12-15T00:00:00Z.PT18H.nc
truth.fc.2009-12-15T00:00:00Z.P1D.nc
truth.fc.2009-12-15T00:00:00Z.P1DT6H.nc
…
truth.fc.2009-12-15T00:00:00Z.P18D.nc
```

You can check that this is true by finding these files in the tree on your left, or list the files in `/home/nonroot/shared/EDU/qgstart/output/truth` with 

In [ ]:
ls $JEDI_EDU/qgstart/output/truth

Remember that we set the initial time in the yaml file as 2009-12-15 00:00:00. `PTXX` means the forecast length from the initial time. E.g., the actual time for the file 2009-12-15T00:00:00Z.PT12H.nc will be 2009-12-15 12:00:00. Check that the total run time is 21 days (look at the files in the list), consistent with your yaml file.<br><br>
We can make a plot of the truth run using the plotting scripts in `/home/nonroot/shared/EDU/plots_scripts`.
You need to use it like this:
```bash
python ./plot.py <model name> <diagnostic to plot> <path/to/file> --plotwind --output <path/to/output/NAME>
```
For example to plot the different fields of the truth run at 16 days, you will need the following commands:

In [ ]:
cd $JEDI_EDU/plots_scripts
mkdir -p $JEDI_EDU/qgstart/output/truth/plots
python ./plot.py qg fields $JEDI_EDU/qgstart/output/truth/truth.fc.2009-12-15T00\:00\:00Z.P16D.nc \
        --plotwind --output $JEDI_EDU/qgstart/output/truth/plots/truth

For more details about the plotting tool, please read [the documentation](https://jointcenterforsatellitedataassimilation-jedi-docs.readthedocs-hosted.com/en/latest/inside/jedi-components/oops/toy-models/toy-models.html#unified-plotting-tool).
<br><br>
This calls python to generate plots for the potential vorticity fields, the stream function fields and the wind fields after 16 days of forecast. You can view these plots in `/home/nonroot/shared/EDU/qgtruth/output/truth/plots/`. The last 2 days will be used as forecasts without data assimilation. These plots will show velocity vectors, if you don't want those you can remove the flag *--plotwind* option in the above instructions.
<br><br>
Now have a look at the output using your favorite .jpg viewer (this can be python). The files `truth_x.jpg` and `truth_q.jpg` contain the stream function (x) and potential vorticity (q) in both layers. These should look something like: 
<br>
<center> <img src="images/truth_x.jpg"/> <img src="images/truth_q.jpg"/>  </center>
<br>
Now we are done with the truth, let's generate the background field in the next step.

# Generate the background state

We now generate our best initial guess of the state of the system, the *background state*. We use the following yaml `file genenspert_B_default.yaml`:

```yaml
background error:
  covariance model: QgError                         # Use the pre-defined QG error model
  horizontal_length_scale: 2.2e6                    # Horizontal correlation scale in m
  maximum_condition_number: 1.0e6
  standard_deviation: 1.8e7                         # Standard deviation in m^2/s
  vertical_length_scale: 15000.0                    # Vertical correlation lengthscale in m
  randomization_seed: 3                             # Random seed
forecast length: PT24H                              # forecast for six days
initial condition:
  date: 2009-12-31T00:00:00Z
  filename: qgstart/output/truth/truth.fc.2009-12-15T00:00:00Z.P15D.nc
members: 1                                         # Generate  perturbed states from the truth 
model:
  name: QG
  tstep: PT1H
output:
  datadir: qgstart/output/bg
  date: 2009-12-30T00:00:00Z
  exp: bkgd
  frequency: PT6H
  type: fc
geometry:
  nx: 40
  ny: 20
  depths: [4000.0, 6000.0]
perturbed variables: [x]                            # Perturb the whole state vector x
```

This file perturbs the state variables `[x]` with a random vector drawn from the **QG Error covariance model** with a **correlation length scale of 2200km and standard deviation 1.8e7 $m^2/s$**. Note that the control run is generated by adding perturbations to the truth at 2009-12-30-00:00:00, and running the QG forecast from there for 1 day.

You can create the background  by running:

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_gen_ens_pert_B.x ./qgstart/yamls/genenspert_B_default.yaml

The output files will be in
`$JEDI_EDU/qgstart/output/bg/`. You can have a look at the perturbation fields and compare the control run (the background you just created) and the truth by running:

In [ ]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg fields \
        $JEDI_EDU/qgstart/output/bg/bkgd.fc.2009-12-30T00\:00\:00Z.P1D.nc \
        $JEDI_EDU/qgstart/output/truth/truth.fc.2009-12-15T00\:00\:00Z.P16D.nc \
        --output $JEDI_EDU/qgstart/output/plots/bkgd_error --title "background error"

The plots will look something like the following. Note that your files might look different because of different random numbers used, but the statistics, such as typical length scales and amplitudes, should be the same. Displayed are the difference between the background stream function and the true stream function for both layers. <br>We can also look at the difference in meridional velocity. Note the size of the differences.
<br>
<center> <img src="images/bkgd_error_x_v_diff.png"/> </center>
<br>

Our next step is to collect observations. In our simulated experiments we actually have to generate the observations from the truth.

# Generate observations

### 1. Generate a single observation

In this tutorial series we will use two types of observation strategies: one observation at one grid point, and 100 observations generated at random locations in the model. We start with the one-observation experiments, using an upper-layer stream function observation.

We need to look at the file `makeobs3d_oneobs.yaml` that contains the settings to generate the observation for our first  experiment. Look at each line and check the comments to see if they make sense to you.
```yaml
geometry:
  nx: 40
  ny: 20
  depths: [4000.0, 6000.0]
state:
  date: 2009-12-31T00:00:00Z
  filename: qgtruth/output/truth/truth.fc.2009-12-15T00:00:00Z.P16D.nc
window begin: 2009-12-30T18:00:00Z
window length: PT12H
observations:
  observers:
  - obs operator:
      obs type: Stream               # The observation type is the stream function
    obs space:
      obsdataout:
        engine:
          obsfile: qgtruth/output/obs/truth_oneob.obs3d.nc
      obs type: Stream
      generate:
        begin: PT6H
        nval: 1
        obs locations:
          lon: [-10]                 # List of longitudes for generated observations (only one here)
          lat: [35]                  # List of latitudes for generated observations (only one here)
          z: [7000.0]                # List of heights for generated observations (only one here)
        obs error: 4.0e6             # Observation error standard deviation, in m^2/s
        obs period: PT12H
make obs: true                       # Generate the observations
obs perturbations: true              # Add random  measurements to the observations
```

Note the last line, in which we add a random error to the observation. This is to mimic what happens in the real world where measurement errors are always present. The measurement error is drawn from a Gaussian distribution with standard deviation given in the line with `obs error: 4.0e6`.

It reads observations at time 18:00, so 18 hours after the start of the truth run.

Execute the following command to make this observation:

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_hofx3d.x ./qgstart/yamls/makeobs3d_oneobs.yaml

The output file is in `/home/nonroot/shared/EDU/qgstart/output/obs/truth_oneob.obs3d.nc`.
To view this file we can convert it to an easily readable txt file with:

In [ ]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg obs $JEDI_EDU/qgstart/output/obs/truth_oneob.obs3d.nc \
        --output $JEDI_EDU/qgstart/output/plots/qg_oneobs

And display the content of this text file (which has the location, the observation value and the truth denoted as hofx, pronounced "H of x"):

In [ ]:
cat $JEDI_EDU/qgstart/output/plots/qg_oneobs.txt

We are now in the same situation as a numerical weather prediction center: we have the background and observations and are ready to perform the data assimilation!

### 2. Generate 100 random observations

We generate 100 observations at random locations using the `makeobs3d_mult_obs.yaml` yaml file

Have a look at the observation-generating part: instead of the following lines in from the single-ob generation in **makeobs3d_oneobs.yaml**:

```yaml
generate:
  begin: PT6H
  nval: 1
  obs locations:
    lon: [-10]                               # List of longitudes for generated observations (only one here)
    lat: [35]                                # List of latitudes for generated observations (only one here)
    z: [7000.0]                              # List of heights for generated observations (only one here)
  obs error: 4.0e6                           # Observation error standard deviation, in m^2/s
  obs period: PT12H
```
We will now use in **makeobs3d_mult_obs.yaml**:
```yaml
generate:
  begin: PT6H
  nval: 1
  obs density: 100                         # Generate 100 observation
  obs error: 4.0e6
  obs period: PT12H
```
Note that we don't have a saying about the locations where the observations are generated.<br>
Run this with:

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_hofx3d.x ./qgstart/yamls/makeobs3d_mult_obs.yaml

You can look quickly at the different positions / values of these observations with

In [ ]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg obs $JEDI_EDU/qgstart/output/obs/truth_100obs.obs3d.nc \
        --output $JEDI_EDU/qgstart/output/plots/qg_multobs
cat $JEDI_EDU/qgstart/output/plots/qg_multobs.txt

# Ensemble background generation
### TBD . . .

That marks the end of the introductory tutorial! We now have **truth, background, and observations** produced. The next step is to perform data assimilation experiments that combine the background and observations with one of the solvers, in order to obtain an analysis that better approximates the truth.

There are several included in this tutorial series: **3Dvar, 4Dvar, LETKF, 3DEnVar, 4DEnVar** 

As users become more advanced, they can also attempt to run **cycling experiments** with 3Dvar or LETKF, which introduces the concept of assimilating observations multiple times at regular intervals, known as the "DA cycling frequency". Cycling helps to better maintain the corrections made by in the DA analyses within the subsequent model forecast.